# RepairScope-X — Notebook 22 | Verification and statistical audit

**CPU-only, fully standalone, reproducible.** No Hugging Face downloads and no GPUs required. The notebook embeds the **694 original records** from the completed Dream, ALoDLM, MDLM and BD3LM experiments (repair trials plus MDLM/BD3LM natural generations), including the **140 previously flagged records**.

### Scope
1. Validate embedded data hashes, unique IDs and provenance.
2. Independently rescore Countdown with a safe arithmetic parser; separate exact correctness from numeric/format proxies.
3. For GSM8K/SVAMP, identify ambiguous answer extraction (`FINAL_ANSWER`, boxed numbers, last number, truncation); **do not assume automatic rescoring is a human ground truth**.
4. Produce a complete machine-generated 140-item reviewer queue with the full outputs and suggested evidence. Reviewers can independently fill a decision CSV and rerun the adjudication stage.
5. Recompute descriptive outcomes and problem-level paired evidence; do **not** treat repeated generation seeds as independent problems.
6. Package raw data, machine metrics, manual template, statistical tables, audit notes and SHA-256 checksums as a ZIP.

**Limitations:** This notebook *prepares* the 140-item human audit and supplies automated checks. It cannot certify that humans reviewed the cases until manual decisions are provided. In-sample oracles remain descriptive upper bounds, not deployable policy estimates.

## 1. Imports (Python standard library only)

In [ ]:
import json, io, csv, hashlib, zlib, base64, re, math, ast, random, zipfile, collections, statistics
from pathlib import Path
from datetime import datetime, timezone
print('Standard-library CPU analysis only — no pip installs required')

## 2. Verified embedded evidence corpus

In [ ]:
DATA_HASH='239645d355bd008e5fc838f4ea356e2c3e7c6b28902e80d809fd7afc0288a202'
DATA_B64='...'
original_bytes=zlib.decompress(base64.b64decode(DATA_B64))
assert hashlib.sha256(original_bytes).hexdigest()==DATA_HASH
DATA=json.loads(original_bytes)
TRIAL_GROUPS=DATA['sources']
PREVIOUS_FLAGS=DATA['prior_review_flags']
PROMPT_INDEX=DATA['prompt_index']
assert sum(len(v) for v in TRIAL_GROUPS.values())==694
assert len(PREVIOUS_FLAGS)==140
for cohort,rows in TRIAL_GROUPS.items():
    assert rows and len({str(r.get('trial_id',r.get('id'))) for r in rows})==len(rows), cohort
    print(cohort, len(rows))
print('Review flags',len(PREVIOUS_FLAGS),'| SHA-256',DATA_HASH)

## 3. Independent safe task verifier and explicit uncertainty

In [ ]:
NUMBER=r'[-+]?(?:\d+(?:\.\d*)?|\.\d+)(?:[eE][-+]?\d+)?'
NUM=re.compile(NUMBER)
FINAL=re.compile(r'(?im)^\s*FINAL_ANSWER\s*:\s*('+NUMBER+r')\s*$',re.I)
BOXED=re.compile(r'\\boxed\s*\{\s*('+NUMBER+r')\s*\}')
PROMPT=re.compile(r'Use the numbers\s+(.+?)\s+exactly once\s+to produce the target\s+('+NUMBER+r')',re.I)
EXPRESSION=re.compile(r'(?<!\w)(?:\(?\s*\d+(?:\.\d+)?\s*\)?\s*[-+*/]\s*){1,8}\(?\s*\d+(?:\.\d+)?\s*\)?')

# Never evaluate arbitrary Python source; only arithmetic AST nodes are allowed.
def arithmetic(expr):
    try: tree=ast.parse(expr,mode='eval')
    except (ValueError,SyntaxError):return None
    literals=[]
    def visit(n):
        if isinstance(n,ast.Expression):return visit(n.body)
        if isinstance(n,ast.Constant) and type(n.value) in (int,float):
            v=float(n.value)
            if not math.isfinite(v):raise ValueError('nonfinite')
            literals.append(v);return v
        if isinstance(n,ast.UnaryOp) and type(n.op) in (ast.USub,ast.UAdd):
            a=visit(n.operand);return -a if type(n.op) is ast.USub else a
        if isinstance(n,ast.BinOp) and type(n.op) in (ast.Add,ast.Sub,ast.Mult,ast.Div):
            a,b=visit(n.left),visit(n.right)
            if type(n.op) is ast.Add:return a+b
            if type(n.op) is ast.Sub:return a-b
            if type(n.op) is ast.Mult:return a*b
            return a/b
        raise ValueError('unsupported AST')
    try:
        value=visit(tree)
        if not math.isfinite(value):return None
        return (value,literals)
    except (ValueError,ZeroDivisionError,OverflowError):return None

def close(a,b):
    return a is not None and b is not None and math.isclose(float(a),float(b),rel_tol=1e-8,abs_tol=1e-8)

def last_number(text):
    nums=NUM.findall(str(text).replace(',',''))
    return float(nums[-1]) if nums else None

def strict_number(text):
    matches=FINAL.findall(str(text))
    return float(matches[-1]) if matches else None

def boxed_number(text):
    matches=BOXED.findall(str(text))
    return float(matches[-1]) if matches else None

def countdown_audit(text,prompt):
    m=PROMPT.search(str(prompt))
    if not m:return dict(verifier_state='missing_prompt',valid=False,candidates=[],required=[],target=None)
    required=[float(x) for x in NUM.findall(m.group(1))]
    goal=float(m.group(2))
    body=str(text).replace('×','*').replace('÷','/').replace('\\times','*')
    candidates=[];found=[]
    for match in EXPRESSION.finditer(body):
        expr=match.group(0).strip()
        checked=arithmetic(expr)
        if checked is None:continue
        value,nums=checked
        same=len(nums)==len(required) and all(close(a,b) for a,b in zip(sorted(nums),sorted(required)))
        is_target=close(value,goal)
        candidates.append(dict(expression=expr,value=value,uses_exact_numbers=same,matches_target=is_target))
        if same and is_target:found.append(expr)
    return dict(verifier_state='checked',valid=bool(found),valid_expression=found[-1] if found else None,
                candidates=candidates[:30],required=required,target=goal)

# Regression tests include benign, altered literals, unrelated target echoes and unsafe code.
assert arithmetic('19+24-29')[0]==14
assert arithmetic('__import__("os").system("echo hi")') is None
assert countdown_audit('19 + 24 - 29 = 14','Use the numbers 19, 29, and 24 exactly once to produce the target 14.')['valid']
assert not countdown_audit('19 + 24 - 28 = 15','Use the numbers 19, 29, and 24 exactly once to produce the target 14.')['valid']
assert not countdown_audit('FINAL_ANSWER: 14','Use the numbers 19, 29, and 24 exactly once to produce the target 14.')['valid']
print('Verifier unit tests passed')

## 4. Standardize 694 records and calculate automated disagreements

In [ ]:
# Normalize each record without altering the original evidence.
def normalize(cohort,r):
    is_repair=cohort not in ('MDLM natural','BD3LM natural')
    result_text=str(r.get('repaired_output',r.get('output','')))
    original_text=str(r.get('original_output',''))
    ds=r.get('dataset','unknown')
    reported=bool(r.get('task_success',r.get('verification',{}).get('task_success',False)))
    strict=strict_number(result_text)
    boxed=boxed_number(result_text)
    numeric=last_number(result_text)
    ver=r.get('verification') or {}
    target=ver.get('target',r.get('target'))
    if target is None:target=ver.get('countdown_structured',{}).get('target')
    target=float(target) if target is not None else None
    candidate=None;status=[];mathcheck=None
    if ds.startswith('countdown'):
        prompt=r.get('prompt') or PROMPT_INDEX.get(r.get('problem_id'))
        if not prompt:
            orig=r.get('original_output','')
            m=PROMPT.search(orig)
            if m:prompt=m.group(0)
        if not prompt and isinstance(ver.get('countdown_structured'),dict):
            co=ver['countdown_structured'];nums=co.get('source_numbers',[])
            if nums and target is not None:
                prompt=f"Use the numbers {', '.join(str(int(n)) if float(n).is_integer() else str(n) for n in nums)} exactly once to produce the target {target}."
        mathcheck=countdown_audit(result_text,prompt or '')
        candidate=bool(mathcheck['valid']) if mathcheck['verifier_state']=='checked' else None
        if candidate is None:status.append('countdown_missing_context')
        if close(numeric,target) and not candidate:status.append('numeric_target_without_valid_expression')
        if candidate is not None and candidate != reported:status.append('independent_countdown_disagreement')
    else:
        if strict is not None: candidate=close(strict,target)
        elif boxed is not None: candidate=close(boxed,target)
        else: candidate=None;status.append('no_explicit_final_answer')
        if boxed is not None and numeric is not None and not close(boxed,numeric):status.append('boxed_last_number_disagree')
        if strict is not None and numeric is not None and not close(strict,numeric):status.append('strict_last_number_disagree')
        if candidate is not None and candidate != reported:status.append('explicit_vs_reported_disagreement')
    if not result_text.strip():status.append('empty_response')
    if len(result_text)<15:status.append('very_short_response')
    return dict(cohort=cohort,kind='repair' if is_repair else 'natural',dataset=ds,
      trial_id=str(r.get('trial_id',r.get('id'))),problem_id=str(r.get('problem_id',r.get('failure_id'))),
      failure_id=str(r.get('failure_id',r.get('problem_id'))),action=r.get('action','natural'),
      reported_success=int(reported),automated_candidate=(int(candidate) if candidate is not None else None),
      target=target,strict_prediction=strict,boxed_prediction=boxed,numeric_prediction=numeric,
      mathcheck=mathcheck,flags=status,output=result_text,original_output=original_text,
      original_record=r)

ALL=[normalize(c,r) for c,rows in TRIAL_GROUPS.items() for r in rows]
assert len(ALL)==694
keyset={(x['cohort'],x['trial_id']) for x in ALL}
assert len(keyset)==694
print('Reported task successes:',sum(x['reported_success'] for x in ALL))
print('Independent high-confidence candidates:',sum(x['automated_candidate']==1 for x in ALL))
print('Records with explicit/structured candidate:',sum(x['automated_candidate'] is not None for x in ALL))
print('Flag types:',collections.Counter(f for x in ALL for f in x['flags']).most_common())

## 5. The original 140-item queue — full-text reviewer template

In [ ]:
# A flagged record is not automatically an error. All previous 140 flags remain visible.
INDEX={(x['cohort'],x['trial_id']):x for x in ALL}
COHORT_MAP={'Dream exploratory':'Dream exploratory','Dream independent':'Dream independent','Dream independent primary':'Dream independent',
            'Dream confirmation secondary':'Dream independent',
            'ALoDLM FP32':'ALoDLM entropy'}
ORIGINAL_QUEUE=[]
for f in PREVIOUS_FLAGS:
    cohort=COHORT_MAP.get(f['cohort'],f['cohort'])
    key=(cohort,f['source_trial_id'])
    a=INDEX.get(key)
    assert a is not None, 'Flag is not mapped to a source trial: '+str(key)
    ORIGINAL_QUEUE.append(dict(cohort=cohort,trial_id=a['trial_id'],problem_id=a['problem_id'],
         dataset=a['dataset'],action=a['action'],original_flags=f['flags'],
         reported_success=a['reported_success'],automated_candidate=a['automated_candidate'],
         new_flags=';'.join(a['flags']),target=a['target'],
         countdown_candidates=json.dumps((a['mathcheck'] or {}).get('candidates',[])[:12],ensure_ascii=False),
         original_output=a['original_output'],repaired_output=a['output'],
         manual_decision='',manual_evidence='',annotator_id=''))
assert len(ORIGINAL_QUEUE)==140
print('140 original flags mapped to exact full-length outputs, without truncating excerpts.')

## 6. Optional adjudications (only applied after human review)

In [ ]:
# Optional: supply a CSV with the same (cohort,trial_id) columns as the reviewer template.
# Supported decision: correct, incorrect, ambiguous. Blank = not reviewed.
# If running in Colab, upload your completed CSV and point OVERRIDE_PATH at it.
OVERRIDE_PATH=''  # example: '/content/RepairScopeX_completed_manual_decisions.csv'
OVERRIDES={}
if OVERRIDE_PATH:
    with open(OVERRIDE_PATH,encoding='utf-8',newline='') as fh:
        for row in csv.DictReader(fh):
            label=row.get('manual_decision','').strip().lower()
            if not label:continue
            assert label in ('correct','incorrect','ambiguous'),label
            key=(row['cohort'],row['trial_id'])
            assert key in INDEX,'Unknown manual trial reference: '+str(key)
            assert key not in OVERRIDES,'Duplicate manual judgement: '+str(key)
            assert row.get('manual_evidence','').strip(), 'Every manual decision requires a short evidence note.'
            OVERRIDES[key]=dict(decision=label,evidence=row['manual_evidence'],annotator=row.get('annotator_id',''))

def audited_value(row):
    label=OVERRIDES.get((row['cohort'],row['trial_id']))
    if label:
        if label['decision']=='correct':return (1,'manual')
        if label['decision']=='incorrect':return (0,'manual')
        return (None,'ambiguous_manual')
    return (row['reported_success'],'original_unreviewed')

ADJUDICATED=[dict(x,audited_success=audited_value(x)[0],evidence_status=audited_value(x)[1]) for x in ALL]
print('Human-reviewed records:',len(OVERRIDES),'of 694; original flagged records:',len(ORIGINAL_QUEUE))
if not OVERRIDES:
    print('No human annotations supplied. Original success numbers remain UNVERIFIED; no manual review is claimed.')

## 7. Cohort statistics and problem-level paired exact tests

In [ ]:
def clopper_pearson_upper_zero(n,alpha=.05):
    # Exact two-sided upper bound for x=0 independent Bernoulli units.
    return 1-(alpha/2)**(1/n)

def exact_sign(positive,negative):
    n=positive+negative
    if n==0:return 1.0
    small=min(positive,negative)
    return min(1.0,2*sum(math.comb(n,k) for k in range(small+1))/2**n)

COHORT_SUMMARY=[]
for (cohort,kind),records in sorted(collections.defaultdict(list,{
    k:[r for r in ADJUDICATED if (r['cohort'],r['kind'])==k]
    for k in {(r['cohort'],r['kind']) for r in ADJUDICATED}
}).items()):
    n=len(records)
    units=len({x['problem_id'] for x in records})
    n_reviewed=sum(x['evidence_status']=='manual' for x in records)
    hits=sum(x['audited_success']==1 for x in records)
    COHORT_SUMMARY.append(dict(cohort=cohort,kind=kind,records=n,unique_problems=units,
      original_successes=sum(x['reported_success'] for x in records),
      currently_counted_successes=hits,manual_reviewed=n_reviewed,manual_ambiguous=sum(x['evidence_status']=='ambiguous_manual' for x in records),
      automated_candidate_available=sum(x['automated_candidate'] is not None for x in records),
      original_vs_automated_disagreements=sum(x['automated_candidate'] is not None and x['automated_candidate']!=x['reported_success'] for x in records)))

PAIR_TESTS=[]
for cohort in sorted({x['cohort'] for x in ADJUDICATED if x['kind']=='repair'}):
    rows=[r for r in ADJUDICATED if r['cohort']==cohort and r['kind']=='repair']
    actions=sorted({r['action'] for r in rows})
    g_name='global'
    if g_name not in actions: continue
    # Average over seeds/traces within each original problem; independently-sampled unit is problem_id.
    by_problem=collections.defaultdict(lambda:collections.defaultdict(list))
    for r in rows:
        if r['audited_success'] is not None:by_problem[r['problem_id']][r['action']].append(r['audited_success'])
    for action in [a for a in actions if a!=g_name]:
        vals=[]
        for p,act_dict in by_problem.items():
            if action in act_dict and g_name in act_dict:
                vals.append(sum(act_dict[action])/len(act_dict[action])-sum(act_dict[g_name])/len(act_dict[g_name]))
        pos=sum(v>1e-12 for v in vals);neg=sum(v< -1e-12 for v in vals);ties=len(vals)-pos-neg
        PAIR_TESTS.append(dict(cohort=cohort,action=action,versus=g_name,paired_problem_ids=len(vals),
            positive=pos,negative=neg,ties=ties,mean_difference=sum(vals)/len(vals) if vals else None,
            exploratory_two_sided_sign_p=exact_sign(pos,neg),
            caveat='Uncorrected for multiple comparisons; only original problem IDs used as independent units.'))
print('Cohort summary',len(COHORT_SUMMARY),'rows; paired tests',len(PAIR_TESTS),'rows')
for x in COHORT_SUMMARY:print(x['cohort'],x['kind'],x['original_successes'],'/',x['records'],'unique problems',x['unique_problems'])
print('Exact 95% upper bound for 0/8 independent problems:',round(clopper_pearson_upper_zero(8),4))

## 8. Export reviewer queue, analysis CSVs, source records and SHA-256 manifest

In [ ]:
# CPU runtime and standard-library only. Artifacts live in /content on Colab, or cwd elsewhere.
OUT=Path('/content/RepairScopeX_Notebook22_Audit') if Path('/content').exists() else Path.cwd()/'RepairScopeX_Notebook22_Audit'
OUT.mkdir(parents=True,exist_ok=True)

def write_csv(path,rows):
    assert rows
    keys=list(rows[0].keys())
    with open(path,'w',encoding='utf-8',newline='') as f:
        w=csv.DictWriter(f,fieldnames=keys,extrasaction='ignore');w.writeheader();w.writerows(rows)

write_csv(OUT/'manual_review_140_full_text.csv',ORIGINAL_QUEUE)
write_csv(OUT/'cohort_summary.csv',COHORT_SUMMARY)
write_csv(OUT/'paired_problem_level_sign_tests.csv',PAIR_TESTS)
export_audit=[]
for row in ADJUDICATED:
    export_audit.append({k:v for k,v in row.items() if k not in ('original_record','mathcheck')})
    export_audit[-1]['flags']=';'.join(row['flags'])
write_csv(OUT/'all_694_record_machine_audit.csv',export_audit)

for cohort,rows in TRIAL_GROUPS.items():
    filename='original_'+re.sub('[^a-z0-9]+','_',cohort.lower()).strip('_')+'.jsonl'
    (OUT/filename).write_text(''.join(json.dumps(r,ensure_ascii=False)+'\n' for r in rows),encoding='utf-8')

(OUT/'source_provenance.json').write_text(json.dumps(DATA['provenance'],indent=2),encoding='utf-8')
(OUT/'audit_gate.json').write_text(json.dumps({
  'embedded_evidence_sha256':DATA_HASH,'total_records':694,'original_review_flags':140,
  'manual_reviews_applied':len(OVERRIDES),'human_verified_complete':len(OVERRIDES)>=140 and all((f['cohort'],f['trial_id']) in OVERRIDES for f in ORIGINAL_QUEUE),
  'audit_generated_utc':datetime.now(timezone.utc).isoformat(),
  'note':'Machine audit is not a substitute for blinded human verification.'},indent=2),encoding='utf-8')

report=['# RepairScope-X Verification Audit',
 f'Embedded original data SHA-256: `{DATA_HASH}`',
 f'Original records: **694**; initial flagged review queue: **140**.',
 f'Manual reviews provided: **{len(OVERRIDES)}**.',
 '',
 '## Rules and caveats',
 '- Countdown automatic candidates require arithmetic correctness using every source number once; unsafe Python evaluation is prohibited.',
 '- GSM8K and SVAMP last-number matching can be wrong if the response contains intermediate calculations; absent explicit final answers remain unadjudicated.',
 '- An original flag is not proof of erroneous scoring. Numeric correctness is not equivalent to Countdown expression validity.',
 '- Original reported task labels are retained in the main cohort analysis until manual adjudication; automated disagreements are flags only.',
 '- 140 entries require a reviewer; all 694 records are embedded and exported for reproducibility.',
 '- Use the `manual_review_140_full_text.csv` template; fill `manual_decision` as correct, incorrect, or ambiguous plus a reason.',
 '- Confidence intervals/paired sign tests operate on problem IDs, not independent seeds; exploratory comparisons have multiplicity caveats.',
 '', '## Source cohort counts']
for cohort,rows in TRIAL_GROUPS.items():report.append(f'- {cohort}: {len(rows)} records')
(OUT/'RESEARCH_AUDIT_NOTES.md').write_text('\n'.join(report)+'\n',encoding='utf-8')
manifest={p.name:hashlib.sha256(p.read_bytes()).hexdigest() for p in OUT.iterdir() if p.is_file()}
(OUT/'sha256_manifest.json').write_text(json.dumps(manifest,indent=2),encoding='utf-8')
ZIP=OUT.parent/'RepairScopeX_NB22_Verification_Audit_results.zip'
with zipfile.ZipFile(ZIP,'w',compression=zipfile.ZIP_DEFLATED) as z:
    for p in sorted(OUT.iterdir()):
        if p.is_file():z.write(p,p.name)
print('Exported',ZIP,'size',ZIP.stat().st_size,'bytes')
if 'google.colab' in __import__('sys').modules:
    from google.colab import files
    files.download(str(ZIP))

## Interpreting the output

The exported `manual_review_140_full_text.csv` contains **all 140 original flagged trial IDs**, their full original/repaired text, diagnostic expressions, and blank adjudication fields. To complete the verification, obtain blinded independent ratings and adjudicate disagreements; then supply the annotated CSV using `OVERRIDE_PATH` and rerun analysis/export cells.

This CPU notebook runs without Notebook 20. After Notebook 20 finishes, its new 128/256-token generation outputs can be audited in a later update, but are not silently included in the frozen corpus used here.